# Orientarsi nel mondo degli anime

Negli ultimi vent'anni, il numero di anime prodotti ogni anno è cresciuto in modo esponenziale. Oggi MyAnimeList cataloga quasi 29.000 titoli, un numero talmente alto da rendere impossibile anche solo sfogliarli tutti. Per un fan, questo non è necessariamente una buona notizia: più contenuto c'è, più diventa difficile trovare quello che vale davvero la pena guardare.

Il problema della scoperta è diventato centrale nell'esperienza del fandom moderno. Non mancano i consigli ma spesso si contraddicono, si sovrappongono, o puntano sempre agli stessi titoli. Il risultato è un paradosso: un catalogo enorme, e la sensazione di non sapere cosa guardare.

MyAnimeList contiene 124 milioni di valutazioni distribuite su quasi 29.000 titoli. Questi dati non servono solo a stilare classifiche ma contengono segnali strutturali che permettono di navigare il catalogo in modo più consapevole.

Questo notebook ne esplora tre:

- **Da dove iniziare?** La mappa dei generi e la Hall of Fame identificano i punti di riferimento del catalogo — i titoli che quasi chiunque conosce e da cui vale la pena partire.
- **La fonte conta?** Da dove nasce un anime? Manga, light novel, storia originale predice sia la qualità media che la probabilità che chi lo inizia lo finisca davvero?
- **Dove cercare oltre il mainstream?** Il catalogo nasconde decine di titoli con score eccellenti e pochissime valutazioni: opere che i meccanismi normali di scoperta non raggiungono mai.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'figure.facecolor': 'white'})

DATA_PATH = '../datasets_cleaned/'

details = pd.read_csv(DATA_PATH + 'details_clean.csv')
stats   = pd.read_csv(DATA_PATH + 'stats_clean.csv')

# Filtro: tipi con contenuto narrativo rilevante
# Special e TV Special inclusi: spesso adattano materiale originale e completano archi narrativi
TIPI = ['TV', 'Movie', 'OVA', 'ONA', 'Special', 'TV Special']
details = details[details['type'].isin(TIPI)].reset_index(drop=True)
stats   = stats[stats['mal_id'].isin(details['mal_id'])].reset_index(drop=True)

print('details:', details.shape)
print('stats:  ', stats.shape)
print('Tipi presenti:', sorted(details['type'].unique()))


details: (24131, 28)
stats:   (24131, 27)
Tipi presenti: ['Movie', 'ONA', 'OVA', 'Special', 'TV', 'TV Special']


## I punti di riferimento: dalla mappa dei generi alla Hall of Fame

Ogni navigazione parte da una visione d'insieme. Prima di scegliere un titolo specifico, vale la pena capire com'è fatto il territorio: quanti generi esistono, quanto sono grandi, quanto sono affidabili in termini di qualità media. Solo dopo aver capito la struttura del catalogo ha senso chiedersi: quali sono i titoli migliori al suo interno?

Il grafico che segue mappa tutti i generi su due assi: quanti anime esistono per quel genere (quanto è popolato il territorio) e qual è il loro punteggio medio (quanto è affidabile come zona di esplorazione). Un genere in alto a destra ha molti titoli di alta qualità: è un territorio ricco e consolidato. Un genere in alto a sinistra ha pochi titoli ma molto buoni: è una nicchia preziosa. Per chi vuole allargare la propria mappa, questo grafico mostra dove vale la pena guardare.

In [9]:
import plotly.express as px

det_g = details.dropna(subset=['genres', 'score']).copy()
det_g['genre'] = det_g['genres'].str[2:-2].str.split("', '")
det_g = det_g.explode('genre')
det_g = det_g[det_g['genre'].str.strip() != '']

genre_stats = (
    det_g.groupby('genre')
    .agg(n_titoli=('mal_id', 'count'), score_medio=('score', 'mean'))
    .query('n_titoli >= 30')
    .sort_values('score_medio', ascending=False)
    .reset_index()
)

score_mean = genre_stats['score_medio'].mean()

fig = px.scatter(
    genre_stats,
    x='n_titoli',
    y='score_medio',
    color='score_medio',
    color_continuous_scale='RdYlGn',
    hover_name='genre',
    hover_data={'n_titoli': True, 'score_medio': ':.2f'},
    labels={
        'n_titoli': 'Numero di titoli nel genere',
        'score_medio': 'Score medio MAL',
    },
    title='<b>La mappa dei generi: volume e qualità media per ogni territorio del catalogo</b>',
)

fig.update_traces(
    marker=dict(size=10, line=dict(color='white', width=1)),
)

# Offset custom per i generi sovrapposti, default per gli altri
custom_offsets = {
    'Girls Love':   dict(xshift=-30, yshift=12),
    'Boys Love':    dict(xshift=40,  yshift=12),
    'Gourmet':      dict(xshift=0,   yshift=-18),
    'Ecchi':        dict(xshift=-25, yshift=0),
    'Slice of Life':dict(xshift=40,  yshift=0),
    'Comedy':       dict(xshift=12,  yshift=-10),
}
default_offset = dict(xshift=12, yshift=9)

for _, row in genre_stats.iterrows():
    off = custom_offsets.get(row['genre'], default_offset)
    fig.add_annotation(
        x=row['n_titoli'],
        y=row['score_medio'],
        text=row['genre'],
        showarrow=False,
        font=dict(size=11),
        **off,
    )

fig.add_hline(
    y=score_mean,
    line_dash='dash',
    line_color='grey',
    opacity=0.5,
    annotation_text='score medio globale',
    annotation_position='top right',
    annotation_font_size=11,
    annotation_font_color='grey',
)

fig.update_layout(
    font_size=12,
    title_font_size=14,
    coloraxis_colorbar=dict(title='Score medio'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='#f0f0f0', zeroline=False),
    yaxis=dict(showgrid=True, gridcolor='#f0f0f0', zeroline=False),
    height=600,
)

fig.show()


Una volta capita la struttura del territorio, la domanda naturale diventa: quali sono i titoli migliori al suo interno? Due criteri diversi danno risposte diverse.

- Il **punteggio medio** aggrega le valutazioni di migliaia di utenti: misura la qualità percepita dalla massa, ovvero quanti spettatori sono rimasti soddisfatti.
- I **favorites** misurano qualcosa di più viscerale: quante persone hanno deciso che quell'anime merita un posto speciale nel proprio profilo, al di là della valutazione numerica.

Le due classifiche che seguono confrontano i top 10 per punteggio e i top 10 per favorites.

In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

top_score = (
    details
    .query('scored_by >= 10000')
    .nlargest(10, 'score')
    [['title', 'score', 'favorites', 'year']]
    .reset_index(drop=True)
)

top_fav = (
    details
    .nlargest(10, 'favorites')
    [['title', 'favorites', 'score']]
    .reset_index(drop=True)
)

in_both   = set(top_score['title']) & set(top_fav['title'])
n_overlap = len(in_both)

COLOR_BOTH = '#F4863E'  # arancione — appare in entrambe le classifiche
COLOR_SOLO = '#5B8DB8'  # blu — solo in questa classifica

def trunc(t, n=18):
    return t[:n] + '...' if len(t) > n else t

fav_titles_r   = top_fav['title'][::-1].tolist()
fav_values_r   = top_fav['favorites'][::-1].tolist()
score_titles_r = top_score['title'][::-1].tolist()
score_values_r = top_score['score'][::-1].tolist()

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=[
        'Top 10 per Favorites',
        'Top 10 per Score',
    ],
    vertical_spacing=0.14,
)

fig.add_trace(go.Bar(
    y=[trunc(t) for t in fav_titles_r],
    x=fav_values_r,
    orientation='h',
    marker_color=[COLOR_BOTH if t in in_both else COLOR_SOLO for t in fav_titles_r],
    hovertext=fav_titles_r,
    hovertemplate='<b>%{hovertext}</b><br>Favorites: %{x:,}<extra></extra>',
    showlegend=False,
), row=1, col=1)

fig.add_trace(go.Bar(
    y=[trunc(t) for t in score_titles_r],
    x=score_values_r,
    orientation='h',
    marker_color=[COLOR_BOTH if t in in_both else COLOR_SOLO for t in score_titles_r],
    hovertext=score_titles_r,
    hovertemplate='<b>%{hovertext}</b><br>Score: %{x:.2f}<extra></extra>',
    showlegend=False,
), row=2, col=1)

# Tracce vuote solo per la legenda
fig.add_trace(go.Bar(
    x=[None], y=[None], orientation='h',
    marker_color=COLOR_BOTH, name='Appare in più classifiche',
))
fig.add_trace(go.Bar(
    x=[None], y=[None], orientation='h',
    marker_color=COLOR_SOLO, name='Solo in questa',
))

fig.update_layout(
    height=800,
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        orientation='h',
        yanchor='top', y=-0.08,
        xanchor='center', x=0.5,
        font_size=12,
    ),
    margin=dict(b=120),
)

fig.update_xaxes(showgrid=True, gridcolor='#f0f0f0', zeroline=False)
fig.update_yaxes(showgrid=False, tickfont_size=11)

# Favorites: asse X parte da 0
fig.update_xaxes(title_text='Numero di Favorites', row=1, col=1)

# Score: asse X parte vicino al minimo per evidenziare le differenze
fig.update_xaxes(
    range=[top_score['score'].min() - 0.3, top_score['score'].max() + 0.15],
    title_text='Score MAL',
    row=2, col=1,
)

fig.show()


La mappa dei generi e la Hall of Fame insieme danno un primo orientamento completo: sappiamo quali territori esistono, qual è la qualità media di ciascuno, e quali titoli rappresentano il meglio in assoluto.

I titoli che compaiono sia nella classifica per punteggio che in quella per favorites sono pochi e questa divergenza è rilevante. Score e favorites misurano fenomeni distinti: il punteggio premia la coerenza qualitativa percepita dal pubblico mentre i favorites misurano l'attaccamento emotivo, quante persone hanno sentito quell'anime come qualcosa di proprio.

Sapere dove si trovano i punti di riferimento è però solo il primo passo. Il vero problema della navigazione emerge subito dopo: adesso che abbiamo visto i titoli al top, quali criteri usare per trovare il prossimo da guardare? Un punto di partenza concreto è la **fonte**: da dove nasce un anime ci dice qualcosa su cosa aspettarsi da esso.

## La fonte conta: da dove nascono gli anime che ami?

Ogni anime inizia come qualcosa d'altro: un manga letto su un treno, un light novel pubblicato su internet, un videogioco che ha venduto milioni di copie, oppure un'idea nata direttamente in uno studio di animazione senza nulla prima di essa.

La fonte non è un dettaglio tecnico — è una variabile che influenza struttura narrativa, ritmo, pubblico di riferimento e, spesso, la qualità del risultato finale. Un manga porta con sé anni di sviluppo dei personaggi già filtrati dal mercato; un light novel ha un pubblico fedele con aspettative precise; un originale non ha nessun termine di paragone, il che è sia un vantaggio che un rischio.

Il grafico che segue risponde a una domanda concreta: guardando la fonte di un anime, possiamo predire qualcosa sulla sua qualità?

In [11]:
# Aggregazione per fonte (min 30 titoli, solo anime con score)
source_agg = (
    details
    .dropna(subset=['source', 'score'])
    .groupby('source')
    .agg(
        n_titoli=('mal_id', 'count'),
        score_medio=('score', 'mean'),
        favorites_medio=('favorites', 'mean'),
    )
    .reset_index()
    .query('n_titoli >= 30')
    .sort_values('n_titoli', ascending=False)
    .reset_index(drop=True)
)

# Palette consistente per tutti i grafici di questa sezione
palette   = px.colors.qualitative.Alphabet[:len(source_agg)]
color_map = dict(zip(source_agg['source'], palette))

source_score = source_agg.sort_values('score_medio', ascending=True)

fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=0.22,
    subplot_titles=['Numero di titoli per fonte', 'Score medio MAL per fonte'],
)

fig.add_trace(go.Bar(
    y=source_agg['source'],
    x=source_agg['n_titoli'],
    orientation='h',
    marker_color=[color_map[s] for s in source_agg['source']],
    hovertemplate='<b>%{y}</b><br>Titoli: %{x:,}<extra></extra>',
    showlegend=False,
), row=1, col=1)

fig.add_trace(go.Bar(
    y=source_score['source'],
    x=source_score['score_medio'].round(2),
    orientation='h',
    marker_color=[color_map[s] for s in source_score['source']],
    hovertemplate='<b>%{y}</b><br>Score medio: %{x:.2f}<extra></extra>',
    showlegend=False,
), row=1, col=2)

fig.update_layout(
    height=520,
    title=dict(
        text='<b>Da dove nascono gli anime: distribuzione e qualità media per fonte</b>',
        x=0.5, font_size=14,
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
)
fig.update_xaxes(showgrid=True, gridcolor='#f0f0f0', zeroline=False, title_text='Titoli', row=1, col=1)
fig.update_xaxes(
    showgrid=True, gridcolor='#f0f0f0', zeroline=False,
    range=[source_score['score_medio'].min() - 0.3, source_score['score_medio'].max() + 0.15],
    title_text='Score medio',
    row=1, col=2,
)
fig.update_yaxes(showgrid=False)
fig.show()



Il grafico a sinistra mostra subito una cosa: il **manga domina in modo schiacciante**. È la fonte di gran lunga più rappresentata nel catalogo — decenni di serializzazione su riviste hanno prodotto un numero enorme di materiale adattabile. Gli anime Original, quelli nati direttamente in studio senza una fonte preesistente, sono numericamente la seconda categoria.

Il grafico a destra racconta però una storia diversa. **Il volume non correla con la qualità media**: le fonti con più titoli non sono necessariamente quelle con lo score più alto. Al contrario, fonti di nicchia come light novel, visual novel e novel tendono ad avere score medi più elevati, probabilmente perché vengono adattate solo quando il materiale di base ha già un pubblico fedele e aspettative alte.

Gli originali mostrano invece la varianza più alta: senza un termine di paragone preesistente, possono essere capolavori assoluti o delusioni complete. Il grafico che segue esplora questa varianza al livello del singolo anime.

In [12]:
det_src = (
    details
    .dropna(subset=['source', 'score', 'scored_by'])
    .query('scored_by >= 0 and source in @source_agg.source.tolist()')
    .copy()
)

fig = px.scatter(
    det_src.sort_values('source'),
    x='scored_by', y='score',
    color='source', color_discrete_map=color_map,
    hover_name='title',
    hover_data={'source': True, 'scored_by': ':,', 'score': ':.2f', 'year': True, 'type': True},
    labels={
        'scored_by': 'Numero di valutazioni (scala log)',
        'score': 'Score MAL',
        'source': 'Fonte',
    },
    title='<b>Qualità e popolarità per fonte: ogni punto è un anime</b>',
    log_x=True, opacity=0.55,
)
fig.update_traces(marker=dict(size=5, line=dict(width=0.3, color='white')))
fig.update_layout(
    height=560,
    plot_bgcolor='white',
    paper_bgcolor='white',
    title_font_size=14,
    xaxis=dict(showgrid=True, gridcolor='#f0f0f0', zeroline=False),
    yaxis=dict(showgrid=True, gridcolor='#f0f0f0', zeroline=False),
    legend=dict(title='Fonte', font_size=11),
)
fig.show()

Il grafico mostra tutti gli anime del dataset su due assi: quanto è stato valutato (scala logaritmica sull'asse X) e quanto è stato giudicato buono (score MAL sull'asse Y). Ogni punto è colorato per fonte. La struttura che emerge non è casuale.

**La nuvola si addensa in basso a sinistra**
La maggior parte dei 15.942 anime rappresentati ha un numero di valutazioni basso, meno di 10.000, e uno score nella fascia 5–7. È il rumore di fondo del catalogo: titoli prodotti, ma mai diventati davvero rilevanti. Il fenomeno è trasversale a tutte le fonti, ma è più marcato per gli Originali e per Unknown.

**Manga: il volume non spiega tutto**
Il Manga è la fonte con più titoli (4.818) e quella che produce quasi tutti i punti nell'angolo in alto a destra — il quadrante dell'eccellenza popolare. I cinque titoli più votati in assoluto sono quasi tutti Manga: *Shingeki no Kyojin* (2.979.733 valutazioni, score 8.56), *Death Note* (2.919.353, score 8.62), *Kimetsu no Yaiba* (2.281.401, score 8.42), *Fullmetal Alchemist: Brotherhood* (2.248.907, score 9.10). Con 580 titoli che superano le 100.000 valutazioni, il Manga è l'unica fonte con una presenza massiccia nella fascia alta su entrambi gli assi. Questo non è un caso: un manga arriva allo studio di animazione già filtrato da anni di serializzazione e da un pubblico fedele che ne ha decretato la rilevanza.

**Light novel e Web novel: qualità senza rumore**
Le Light novel hanno lo score medio più alto tra le fonti numericamente significative. È il segnale di una fonte che viene adattata con selettività, non ogni light novel diventa un anime, solo quelle con un fandom già consolidato. Il risultato è visibile nel grafico: i punti Light novel si concentrano nella fascia medio-alta dello score con una dispersione contenuta. I Web novel vanno ancora oltre per qualità media, ma sono solo 151 titoli rendendoli una nicchia ancora più selettiva.

**Original: la fonte più rischiosa**
Gli Originali sono la fonte con il range più ampio: dallo score di 2.01 di *Utsu Musume Sayuri* all'8.91 di *Code Geass: Hangyaku no Lelouch R2*. Non c'è filtro preesistente: nessun fandom di lettori, nessuna serializzazione. Il risultato è la distribuzione più dispersa del grafico: molti punti in basso a sinistra, pochi in alto ma potenzialmente altissimi. Tra i titoli che hanno raggiunto il quadrante dell'eccellenza ci sono *Kimi no Na wa.* (8.83), *Sen to Chihiro no Kamikakushi* (8.77) e *Cowboy Bebop* (8.75) tutti partiti senza una base di lettori, tutti diventati punti di riferimento.

**Unknown: la zona morta**
I titoli con fonte Unknown hanno lo score medio più basso (5.89) e non raggiungono mai più di 22.707 valutazioni. La mancanza di informazione sulla fonte non è casuale: riflette anime prodotti in contesti marginali, distribuiti in modo limitato, spesso senza un materiale di base strutturato. La loro posizione nel grafico in basso e a sinistra è coerente con questa lettura.

**Quello che il grafico non mostra**
La scala logaritmica sull'asse X comprime le distanze reali: la differenza tra 1.000 e 10.000 valutazioni occupa lo stesso spazio visivo di quella tra 100.000 e 1.000.000. Questo significa che l'angolo in alto a destra è molto più esclusivo di quanto sembri: meno dell'1% dei titoli nel dataset raggiunge contemporaneamente uno score ≥ 8.5 e più di 500.000 valutazioni e quasi tutti vengono dal Manga.


## Il rischio di abbandonare: quali anime si finiscono davvero?

Ogni anime che si inizia può finire in tre modi: completato, abbandonato, o lasciato in sospeso. MAL traccia esattamente questo: per ogni titolo sa quanti utenti lo hanno finito, quanti lo hanno abbandonato a metà, e quanti stanno ancora guardando.

Questo dato apre una prospettiva diversa sulla navigazione. Non basta sapere che un genere ha uno score alto — bisogna anche sapere se gli anime di quel genere si finiscono. Un titolo con score 8.5 ma drop rate del 25% è un'avvertenza: molte persone ci hanno provato e hanno smesso. Uno con score 7.5 e drop rate del 5% è un segnale opposto: chi lo inizia quasi sempre lo porta a termine.

La heatmap che segue mostra il **tasso di abbandono medio** — `dropped / (watching + completed + on_hold + dropped)` — per ogni combinazione di genere e fonte. I valori alti indicano territori rischiosi; i valori bassi indicano anime che tendono a tenere il pubblico fino alla fine.

In [13]:
import plotly.graph_objects as go

det_genres = details.dropna(subset=['genres']).copy()
det_genres['genre'] = det_genres['genres'].str[2:-2].str.split("', '")
det_genres = det_genres.explode('genre')
det_genres = det_genres[det_genres['genre'].str.strip() != '']

top_genres = det_genres['genre'].value_counts().head(12).index.tolist()
main_sources = ['Manga', 'Light novel', 'Visual novel', 'Novel', 'Web manga', 'Game', 'Original']

merged = (
    stats
    .assign(
        engaged=lambda d: d['watching'] + d['completed'] + d['on_hold'] + d['dropped'],
        drop_rate=lambda d: d['dropped'] / d['engaged'].replace(0, float('nan')),
    )
    .merge(details[['mal_id', 'source', 'genres']], on='mal_id')
    .assign(genre=lambda d: d['genres'].str[2:-2].str.split("', '"))
    .explode('genre')
    .pipe(lambda d: d[d['genre'].str.strip() != ''])
)

pivot = (
    merged
    .query('genre in @top_genres and source in @main_sources')
    .groupby(['source', 'genre'])['drop_rate']
    .mean()
    .unstack(fill_value=float('nan'))
    .reindex(main_sources)[top_genres]
    * 100
)

genre_order = pivot.mean(axis=0).sort_values(ascending=False).index
pivot = pivot[genre_order]

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='YlOrRd',
    zmin=4, zmax=25,
    text=pivot.round(1).astype(str).values,
    texttemplate='%{text}%',
    textfont=dict(size=11),
    hovertemplate='<b>%{y} — %{x}</b><br>Tasso di abbandono: %{z:.1f}%<extra></extra>',
    colorbar=dict(title='Abbandono (%)', ticksuffix='%'),
))

fig.update_layout(
    title=dict(
        text='<b>Tasso di abbandono per genere e fonte: quali anime si finiscono davvero?</b>',
        font_size=14,
    ),
    xaxis=dict(title='Genere', tickangle=-35),
    yaxis=dict(title='Fonte', autorange='reversed'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=380,
)

fig.show()


La heatmap ordina i generi da sinistra a destra per tasso di abbandono medio decrescente. Si nota un pattern che indica che il tasso di abbandono dipende più dalla fonte piuttosto che dal genere.

**Light novel: la fonte più sicura**
La riga Light novel è la più fredda della heatmap su quasi tutti i generi. Il motivo è strutturale: solo una piccola frazione delle light novel viene adattata, e lo studio sceglie quelle con una fanbase già consolidata. Chi inizia un anime da light novel sa già, spesso, cosa sta andando a guardare e lo finisce.

**Original: la fonte più rischiosa**
La riga Original è la più calda. Senza un materiale di base che abbia già filtrato il pubblico, gli originali partono senza fanbase. Il dato sul Fantasy è il più alto dell'intera heatmap: l'isekai, genere dominante tra gli originali degli ultimi anni, è esattamente il tipo di serie che attira molti spettatori curiosi ma ne trattiene pochi fino alla fine.

**Manga: uniforme e affidabile**
Il Manga ha la distribuzione più piatta. Decenni di serializzazione su riviste hanno già selezionato il materiale: chi arriva all'adattamento anime ha superato anni di giudizio del pubblico lettore. Il rischio rimane basso indipendentemente dal genere.

**Fantasy: il genere più abbandonato**
È il genere con la colonna più calda per quasi tutte le fonti. L'eccezione significativa è il Manga: i manga Fantasy che vengono adattati sono quasi tutti titoli consolidati con fanbase fedeli. Il problema del Fantasy è concentrato negli originali e nelle web novel, dove il genere è dominato da isekai di qualità variabile.

**Romance ed Ecchi: i più "safe"**
Indipendentemente dalla fonte, Romance ed Ecchi hanno i valori più bassi. Chi sceglie consapevolmente questi generi tende ad avere aspettative chiare e a portare a termine quello che inizia. Non è un giudizio di qualità ma una misura di coerenza tra aspettativa e risultato.

**Game: il Drama quasi non si abbandona**
Il Drama da Game segna 5.9%, il valore più basso di tutta la colonna Drama. I videogiochi con una componente narrativa forte costruiscono un attaccamento emotivo ai personaggi che si trasferisce nell'adattamento anime: chi arriva alla serie conosce già la storia e raramente abbandona.


## Oltre la mappa conosciuta: i capolavori nascosti

Classifiche e fonte insieme disegnano una mappa densa nella zona dei titoli popolari. Gran parte del catalogo anime è territorio inesplorato: titoli prodotti in nicchie, distribuiti in mercati limitati, usciti in anni in cui la comunità internazionale era più piccola, o semplicemente mai finiti nel flusso delle conversazioni mainstream.

MAL permette d'identificare questi titoli con precisione. Un anime con score ≥ 8.0 è considerato eccellente da chi lo ha visto: la media MAL si aggira intorno al 6.4, e superare l'8.0 richiede un consenso positivo significativo. La popolarità si misura con `scored_by`, quante persone hanno effettivamente valutato l'anime, che funziona indipendentemente dalla lunghezza della serie o dal suo stato. Un titolo con `scored_by` basso e magari un punteggio alto non viene visualizzato nei grafici standard, il che rende difficile la scoperta di queste gemme nascoste.

Per visualizzare questi titoli impostiamo questo criterio: score ≥ 8.0 e scored_by tra 200 e 5.000. La soglia inferiore garantisce che il punteggio non sia il risultato di un campione troppo piccolo e quella superiore esclude i titoli già conosciuti dalla massa. Il risultato è una lista di titoli per chi ha già esaurito le opere più mainstream.

In [14]:
import plotly.express as px

hidden_all = (
    details
    .query('score >= 8.0 and scored_by >= 200 and scored_by < 5000')
    .sort_values('score', ascending=False)
    .reset_index(drop=True)
)

rarest  = hidden_all.loc[hidden_all['scored_by'].idxmin(), 'title']
best    = hidden_all.iloc[0]['title']
n_found = len(hidden_all)

fig = px.scatter(
    hidden_all,
    x='scored_by',
    y='score',
    color='source',
    hover_name='title',
    hover_data={'source': True, 'scored_by': ':,', 'score': ':.2f', 'year': True, 'type': True},
    labels={
        'scored_by': 'Numero di valutazioni  ←  più raro   /   più conosciuto  →',
        'score': 'Score MAL',
        'source': 'Fonte',
    },
    title=f'<b>{n_found} capolavori nascosti: score ≥ 8.0, meno di 5.000 valutazioni</b>',
)

fig.update_traces(
    marker=dict(size=11, line=dict(width=0.8, color='white')),
)

fig.update_layout(
    height=580,
    plot_bgcolor='white',
    paper_bgcolor='white',
    title_font_size=14,
    xaxis=dict(showgrid=True, gridcolor='#f0f0f0', zeroline=False),
    yaxis=dict(showgrid=True, gridcolor='#f0f0f0', zeroline=False),
    legend=dict(title='Fonte', font_size=11),
)

fig.show()


Lo scatter mostra i 29 titoli che soddisfano entrambi i criteri: score ≥ 8.0 e meno di 5.000 valutazioni. Sull'asse X c'è la rarità, più un punto è a sinistra, meno persone lo hanno visto. Sull'asse Y c'è la qualità. Il quadrante in alto a sinistra è la zona più preziosa: opere eccellenti che quasi nessuno ha ancora trovato.

**Il titolo più nascosto**
*Fanren Xiu Xian Chuan: Waihai Fengyun* ha solo 542 valutazioni eppure supera l'8.0. È il punto più a sinistra del grafico. Tre titoli in tutto si trovano sotto le 1.000 valutazioni mantenendo uno score ≥ 8.0: sono le destinazioni più remote del catalogo.

**Il top per qualità**
*Chainsaw Man Movie: Reze-hen* è il titolo con lo score più alto (8.94) ma anche quello più vicino alla soglia delle 5.000 valutazioni (4.504). Con soli 3 titoli sopra 8.5, il filtro è effettivamente severo in quanto superare quella soglia con così poche valutazioni richiede un consenso quasi unanime tra chi l'ha visto.

**Una geografia inaspettata**
Il dato più sorprendente del grafico è la distribuzione per fonte: 19 dei 29 titoli vengono da Novel e Web novel, quasi tutti adattamenti di serie cinesi distribuiti come ONA. Le piattaforme cinesi producono contenuto che non raggiunge quasi mai il pubblico internazionale, ma chi lo trova lo valuta molto in alto. Il Manga, che domina il catalogo generale, contribuisce con solo 2 titoli a questa lista.

**Perché `scored_by` e non i completamenti**
Usare `scored_by` invece dei completamenti (con status completed) risolve un problema strutturale: serie lunghe come *One Piece* o *Naruto* hanno pochissimi completamenti semplicemente perché finirle richiede anni, non perché siano sconosciute. `scored_by` cattura chiunque abbia guardato abbastanza episodi da esprimere una valutazione, indipendentemente dalla lunghezza.

## Quello che i dati ci dicono su come orientarsi in un catalogo sconfinato

La **fonte** è il segnale più robusto. Predice la qualità media, predice la popolarità che un titolo raggiungerà e predice anche quante persone lo porteranno a termine. Una light novel adattata arriva con anni di selezione già incorporati: il pubblico che la aspettava esiste, sa cosa vuole, e finisce quello che inizia. Un originale parte senza questa storia: può diventare un capolavoro o perdersi nel rumore. La fonte non garantisce nulla, ma sposta le probabilità in modo misurabile.

Le **hidden gems** rivelano un catalogo parallelo che i meccanismi normali di scoperta non raggiungono. I 29 titoli che superano 8.0 con meno di 5.000 valutazioni non sono distribuiti a caso: sono quasi tutti Novel e Web novel, quasi tutti ONA cinesi, quasi tutti invisibili alle community internazionali.

Il **tasso di abbandono** aggiunge la dimensione del rischio che score e popolarità non catturano. Un anime con score alto ma drop rate elevato è un avvertimento: molte persone ci hanno provato e si sono fermate. Incrociare i due segnali di qualità percepita da chi ha finito e probabilità di arrivare alla fine dà un'immagine più onesta di cosa vale la pena iniziare. Fantasy e Adventure da fonti originali sono il territorio più rischioso; Light novel su qualsiasi genere sono il più sicuro.